# Exploratory Data Analysis (EDA)

Identificar empíricamente los umbrales geométricos que definen la posesión del balón.

## Fase 0: Preparar Enviroment y Cargar Datos

First of all we are going to import the needed libraries.

In [1]:
! pip install kloppy

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Manejo de datos y operaciones matemáticas
import pandas as pd
import numpy as np

# Visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid") # Configuración estética de Seaborn

# Procesamiento de imágenes/vídeo
import cv2

# Lectura de metadatos y archivos serializados
import json
import pickle
import os, sys # Útil para manejar las rutas 

# Para cargar datos
from kloppy import sportscode, metrica
sys.path.append('..')
import games

Now we are going to load our data.

In [3]:
# Load Tagging
sys.path.insert(0, os.path.abspath('..'))
print("Cargando Smart Tagging (Eventos)...")
path = games.asset_paths()['smart_tagging']
events_df = sportscode.load(path).to_df()
print(f"-> {len(events_df)} instancias cargadas con éxito.")
display(events_df.head())

Cargando Smart Tagging (Eventos)...
-> 638 instancias cargadas con éxito.


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,None,None
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,None,None
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,None,None


In [4]:
# Load ATD
sys.path.insert(0, os.path.abspath('..'))

print("Cargando Tracking ATD...")
paths = games.asset_paths(source='tactical')

dataset = metrica.load_tracking_epts(
    meta_data=paths['atd_meta'],
    raw_data=paths['atd_raw'], 
)

print(f"-> {len(dataset.records)} frames cargados con éxito.")
frame_df = dataset.to_df()
display(frame_df.head())

Cargando Tracking ATD...
-> 199319 frames cargados con éxito.


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,1_x,...,44_d,44_s,45_x,45_y,45_d,45_s,46_x,46_y,46_d,46_s
0,1,0 days 00:00:00.040000,1,None,None,NaN,NaN,None,None,0.33411,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
1,1,0 days 00:00:00.080000,2,None,None,NaN,NaN,None,None,0.33404,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
2,1,0 days 00:00:00.120000,3,None,None,NaN,NaN,None,None,0.33396,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
3,1,0 days 00:00:00.160000,4,None,None,NaN,NaN,None,None,0.33389,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN
4,1,0 days 00:00:00.200000,5,None,None,NaN,NaN,None,None,0.33382,...,None,NaN,NaN,NaN,None,NaN,0.06057,0.39965,None,NaN


In [5]:
# Load Homography
print("Cargando Homografía...")
sys.path.append('../starter') 
import load_homography
matrices, reprojection_error, valid = load_homography.load(source='tactical')
print(f"-> {len(matrices)} matrices cargadas.")

Cargando Homografía...
-> 199319 matrices cargadas.


## Fase 1: Auditoría de Calidad y Ruido (Tracking Crudo)